# Scraping Elecciones Colombia — 8 de marzo de 2026

Extracción estructurada de datos electorales desde el portal de resultados de la Registraduría Nacional.

**Fuente**: `https://resultados.registraduria.gov.co`

> **Nota técnica**: La cookie `cf_clearance` (protección Cloudflare) puede vencer en horas o días. Si recibes un error `403 Forbidden`, copia una nueva cookie desde el navegador y actualiza el archivo `config.json` en la raíz del proyecto.

In [1]:
import json
import os
import requests
import pandas as pd

# Cargar configuración centralizada (cookies, headers, URLs)
CONFIG_PATH = os.path.join('..', '..', 'config.json')
with open(CONFIG_PATH) as f:
    config = json.load(f)

URL = config['nomenclator_url']
cookies = config['cookies']
headers = config['headers']

response = requests.get(URL, cookies=cookies, headers=headers)

if response.status_code == 200:
    data = response.json()
    print(f"✅ nomenclator.json cargado ({len(data['elec'])} elecciones, {len(data['partidos'])} partidos)")
else:
    print("❌ Request failed with status code:", response.status_code)
    print("   → Verifica que cf_clearance en config.json sea válido")

✅ nomenclator.json cargado (4 elecciones, 348 partidos)


# Paso 1: Entender la estructura del JSON

El archivo `nomenclator.json` es el **diccionario maestro** de las elecciones. La Registraduría lo usa para que los archivos de resultados sean livianos: en vez de enviar nombres completos, envía códigos numéricos. Este JSON nos permite "traducir" esos códigos a nombres reales.

**Llaves principales del JSON:**
| Llave | Descripción |
|-------|-------------|
| `ver` | Versión del archivo |
| `y` | Año electoral (2026) |
| `elec` | Lista de **elecciones/corporaciones** (Senado, Cámara, Consultas, CITREP) con sus circunscripciones |
| `levels` | **Niveles** de la jerarquía geográfica (Colombia → Departamento → Municipio → Zona → Comuna → Puesto → Mesa) |
| `amb` | **Ámbitos** (División Político-Administrativa / Divipol). Contiene todos los lugares organizados jerárquicamente |
| `partidos` | Lista de **partidos políticos** con código, nombre y color |

A continuación construiremos un DataFrame (tabla de dimensión) para cada entidad.

In [2]:
# Verificamos las llaves principales del JSON
print("Llaves principales del nomenclator:")
for key in data.keys():
    val = data[key]
    tipo = type(val).__name__
    if isinstance(val, list):
        print(f"  '{key}' -> {tipo}, {len(val)} elementos")
    elif isinstance(val, dict):
        print(f"  '{key}' -> {tipo}")
    else:
        print(f"  '{key}' -> {tipo}: {val}")

Llaves principales del nomenclator:
  'ver' -> int: 1
  'y' -> str: 2026
  'elec' -> list, 4 elementos
  'levels' -> list, 7 elementos
  'amb' -> list, 4 elementos
  'partidos' -> list, 348 elementos


## Paso 2: Tabla de dimensión — Elecciones y Circunscripciones (`dim_elecciones`)

Cada elección (Senado, Cámara, etc.) tiene una o más **circunscripciones** (también llamadas "cámaras" en el JSON, campo `cam`). Por ejemplo, el Senado tiene circunscripción Nacional e Indígena, mientras que Cámara tiene Territorial Departamental, Indígena y Afro.

Creamos una tabla plana donde cada fila es una combinación **elección + circunscripción**.

In [3]:
# Construimos dim_elecciones: una fila por cada combinación elección + circunscripción
filas_elecciones = []

for eleccion in data['elec']:
    for cam in eleccion['cam']:
        filas_elecciones.append({
            'elec_id': eleccion['i'],          # Índice interno de la elección
            'elec_codigo': eleccion['elec'],    # Código de la elección usado en los archivos de votos
            'elec_orden': eleccion['ord'],      # Orden de presentación
            'elec_sigla': eleccion['sigla'],    # Sigla (SE, CA, CN, CT)
            'elec_nombre': eleccion['n'],       # Nombre completo de la elección
            'elec_color': eleccion['c'],        # Color principal
            'elec_color_sec': eleccion['cs'],   # Color secundario
            'cam_id': cam['i'],                 # Índice de la circunscripción
            'cam_nombre': cam['n'],             # Nombre de la circunscripción
            'cam_color': cam['c'],              # Color de la circunscripción
        })

dim_elecciones = pd.DataFrame(filas_elecciones)
print(f"dim_elecciones: {len(dim_elecciones)} filas")
dim_elecciones

dim_elecciones: 7 filas


,elec_id,elec_codigo,elec_orden,elec_sigla,elec_nombre,elec_color,elec_color_sec,cam_id,cam_nombre,cam_color
0,0,1,1,SE,SENADO,#1C283D,#47658F,0,NACIONAL,#FFC53D
1,0,1,1,SE,SENADO,#1C283D,#47658F,4,INDIGENAS,#004F9A
2,1,2,2,CA,CAMARA,#1C283D,#919662,1,TERRITORIAL DEPARTAMENTAL,#FFC53D
3,1,2,2,CA,CAMARA,#1C283D,#919662,4,INDIGENAS,#004F9A
4,1,2,2,CA,CAMARA,#1C283D,#919662,5,AFRO-DESCENDIENTES,#EC0000
5,2,6,3,CN,CONSULTAS INTERPARTIDISTAS,#1C283D,#64a70b,0,NACIONAL,#FFC53D
6,3,7,4,CT,CITREP,#1C283D,#925862,9,CITREP,#A2789C


## Paso 3: Tabla de dimensión — Niveles Geográficos (`dim_niveles`)

La jerarquía geográfica de la Registraduría tiene 7 niveles. Esta tabla es pequeña pero crucial para entender qué nivel (`l`) corresponde a qué tipo de entidad territorial.

`1: COLOMBIA → 2: DEPARTAMENTO → 3: MUNICIPIO → 4: ZONA → 5: COMUNA → 6: PUESTO → 7: MESA`

In [4]:
# Construimos dim_niveles: directo del array 'levels'
dim_niveles = pd.DataFrame(data['levels'])
dim_niveles.columns = ['nivel_id', 'nivel_nombre']

print(f"dim_niveles: {len(dim_niveles)} filas")
dim_niveles

dim_niveles: 7 filas


,nivel_id,nivel_nombre
0,1,COLOMBIA
1,2,DEPARTAMENTO
2,3,MUNICIPIO
3,4,ZONA
4,5,COMUNA
5,6,PUESTO
6,7,MESA


## Paso 4: Tabla de dimensión — Partidos Políticos (`dim_partidos`)

Cada partido tiene un código (`codpar`) que se usa en los archivos de resultados de votos. Esta tabla nos permite traducir ese código al nombre completo del partido y su color oficial.

Hay **348 registros** que incluyen partidos, movimientos, coaliciones y categorías especiales como "Votos en blanco, nulos y no marcados".

In [5]:
# Construimos dim_partidos: directo del array 'partidos'
dim_partidos = pd.DataFrame(data['partidos'])

# Renombramos columnas a nombres descriptivos
dim_partidos.rename(columns={
    'codpar': 'partido_codigo',
    'nombre': 'partido_nombre',
    'color': 'partido_color',
    'i': 'partido_indice',
    's': 'partido_slug',
}, inplace=True)

print(f"dim_partidos: {len(dim_partidos)} filas")
dim_partidos.head(10)

dim_partidos: 348 filas


,partido_codigo,partido_nombre,partido_color,partido_indice,partido_slug
0,0,"VOTOS EN BLANCO, NULOS Y NO MARCADOS",#5A7CCC,1,"VOTOS-EN-BLANCO,-NULOS-Y-NO-MARCADOS"
1,1,PARTIDO LIBERAL COLOMBIANO,#E30716,2,PARTIDO-LIBERAL-COLOMBIANO
2,2,PARTIDO CONSERVADOR COLOMBIANO,#0867B1,3,PARTIDO-CONSERVADOR-COLOMBIANO
3,3,PARTIDO CAMBIO RADICAL,#E3051C,4,PARTIDO-CAMBIO-RADICAL
4,4,PARTIDO ALIANZA VERDE,#007C34,5,PARTIDO-ALIANZA-VERDE
5,5,"MOVIMIENTO AUTORIDADES INDÍGENAS DE COLOMBIA ""...",#C05B16,6,"MOVIMIENTO-AUTORIDADES-INDIGENAS-DE-COLOMBIA-""..."
6,6,"PARTIDO ALIANZA SOCIAL INDEPENDIENTE ""ASI""",#F7C412,7,"PARTIDO-ALIANZA-SOCIAL-INDEPENDIENTE-""ASI"""
7,7,PARTIDO POLÍTICO MIRA,#06539C,8,PARTIDO-POLITICO-MIRA
8,8,PARTIDO DE LA UNIÓN POR LA GENTE - PARTIDO DE ...,#48AB38,9,PARTIDO-DE-LA-UNION-POR-LA-GENTE---PARTIDO-DE-...
9,11,PARTIDO CENTRO DEMOCRÁTICO,#1E477D,10,PARTIDO-CENTRO-DEMOCRATICO


## Paso 5: Tabla de dimensión — Ámbitos Geográficos / Divipol (`dim_ambitos`)

Esta es la tabla más importante y compleja. El array `amb` contiene la **División Político-Administrativa (Divipol)** organizada por elección. Cada ámbito tiene:

| Campo | Significado |
|-------|-------------|
| `i` | Índice interno (posición en el array) |
| `n` | Nombre del lugar |
| `co` | Código Divipol de la Registraduría |
| `s` | Slug (nombre para URL) |
| `l` | Nivel jerárquico (1=Colombia, 2=Departamento, 3=Municipio) |
| `p` | **Padres**: lista de índices que apuntan al nivel superior |
| `h` | **Hijos**: lista de índices que apuntan al nivel inferior |
| `r` | Array de resoluciones/referencias internas |

Vamos a extraer una tabla plana de ámbitos para cada elección, resolviendo la relación padre para saber a qué departamento pertenece cada municipio.

In [6]:
# Primero exploramos cuántos ámbitos hay por elección y por nivel
for grupo in data['amb']:
    elec_id = grupo['elec']
    ambitos = grupo['ambitos']
    
    # Conteo por nivel
    conteo = {}
    for a in ambitos:
        nivel = a['l']
        conteo[nivel] = conteo.get(nivel, 0) + 1
    
    print(f"Elección {elec_id}: {len(ambitos)} ámbitos totales")
    for nivel in sorted(conteo.keys()):
        nombre_nivel = next((l['n'] for l in data['levels'] if l['l'] == nivel), '?')
        print(f"  Nivel {nivel} ({nombre_nivel}): {conteo[nivel]}")
    print()

Elección 1: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 2: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 6: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 7: 185 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 16
  Nivel 3 (MUNICIPIO): 168



In [7]:
# Verificamos si las elecciones 1, 2 y 6 comparten los mismos ámbitos
# Comparamos los códigos (co) y nombres (n) de los ámbitos
ambitos_por_elec = {}
for grupo in data['amb']:
    codigos = set((a['co'], a['n'], a['l']) for a in grupo['ambitos'])
    ambitos_por_elec[grupo['elec']] = codigos

# Comparar elección 1 vs 2
iguales_1_2 = ambitos_por_elec[1] == ambitos_por_elec[2]
iguales_1_6 = ambitos_por_elec[1] == ambitos_por_elec[6]
print(f"¿Elecciones 1 y 2 tienen los mismos ámbitos? {iguales_1_2}")
print(f"¿Elecciones 1 y 6 tienen los mismos ámbitos? {iguales_1_6}")
print(f"Elección 7 (CITREP) tiene un subconjunto de {len(ambitos_por_elec[7])} ámbitos")

¿Elecciones 1 y 2 tienen los mismos ámbitos? True
¿Elecciones 1 y 6 tienen los mismos ámbitos? True
Elección 7 (CITREP) tiene un subconjunto de 185 ámbitos


### Construcción de `dim_ambitos`

Las elecciones 1 (Senado), 2 (Cámara) y 6 (Consultas) comparten **exactamente los mismos 1224 ámbitos**. La elección 7 (CITREP) usa un subconjunto de 185.

Tomamos los ámbitos de la elección 1 como base y resolvemos la relación padre-hijo para crear una tabla donde cada municipio tenga su departamento asociado.

**Lógica de resolución del padre**: El campo `p` de cada ámbito contiene una lista de objetos `{l: nivel, p: [índices]}`. Para un municipio (nivel 3), su padre es el departamento (nivel 2) cuyo índice está en `p[0].p[0]`.

In [ ]:
# Tomamos los ámbitos de la elección 1 (Senado) como base completa
ambitos_base = data['amb'][0]['ambitos']  # elec=1

# Paso 1: Crear un diccionario de lookup por índice para resolver padres
lookup = {a['i']: a for a in ambitos_base}

# Paso 2: Construir filas planas resolviendo la jerarquía
filas_ambitos = []

for a in ambitos_base:
    fila = {
        'ambito_indice': a['i'],            # Índice interno (posición en el array)
        'ambito_nombre': a['n'],             # Nombre del lugar
        'ambito_codigo': a['co'],            # Código Divipol
        'ambito_slug': a.get('s', ''),       # Slug para URL
        'nivel_id': a['l'],                  # Nivel jerárquico
    }
    
    # Resolver el padre inmediato (si existe)
    # El campo 'p' es una lista de relaciones: [{l: nivel_padre, p: [indices_padre]}]
    padre_indice = None
    if a['p']:
        # Tomamos la primera relación padre
        padre_rel = a['p'][0]
        if padre_rel['p']:
            padre_indice = padre_rel['p'][0]
    
    fila['padre_indice'] = padre_indice
    
    # Si es un municipio (nivel 3), resolver el nombre del departamento padre
    if a['l'] == 3 and padre_indice is not None:
        padre = lookup.get(padre_indice, {})
        fila['departamento_nombre'] = padre.get('n', '')
        fila['departamento_codigo'] = padre.get('co', '')
    elif a['l'] == 2:
        # El departamento es su propio departamento
        fila['departamento_nombre'] = a['n']
        fila['departamento_codigo'] = a['co']
    else:
        fila['departamento_nombre'] = None
        fila['departamento_codigo'] = None
    
    filas_ambitos.append(fila)

dim_ambitos = pd.DataFrame(filas_ambitos)
print(f"dim_ambitos: {len(dim_ambitos)} filas")
print("\nDistribución por nivel:")
print(dim_ambitos.groupby('nivel_id').size().reset_index(name='cantidad'))
dim_ambitos.head(10)

dim_ambitos: 1224 filas

Distribución por nivel:
   nivel_id  cantidad
0         1         1
1         2        34
2         3      1189


,ambito_indice,ambito_nombre,ambito_codigo,ambito_slug,nivel_id,padre_indice,departamento_nombre,departamento_codigo
0,0,COLOMBIA,00,COLOMBIA,1,NaN,NaN,NaN
1,1,AMAZONAS,6000,AMAZONAS,2,0.0,AMAZONAS,6000
2,2,ANTIOQUIA,0100,ANTIOQUIA,2,0.0,ANTIOQUIA,0100
3,3,ARAUCA,4000,ARAUCA,2,0.0,ARAUCA,4000
4,4,ATLANTICO,0300,ATLANTICO,2,0.0,ATLANTICO,0300
5,5,BOGOTA D.C.,1600,BOGOTA-D.C.,2,0.0,BOGOTA D.C.,1600
6,6,BOLIVAR,0500,BOLIVAR,2,0.0,BOLIVAR,0500
7,7,BOYACA,0700,BOYACA,2,0.0,BOYACA,0700
8,8,CALDAS,0900,CALDAS,2,0.0,CALDAS,0900
9,9,CAQUETA,4400,CAQUETA,2,0.0,CAQUETA,4400


## Paso 6: Tablas de conveniencia — Departamentos y Municipios

A partir de `dim_ambitos` creamos dos vistas filtradas que son las más útiles para análisis:

- **`dim_departamentos`**: Solo los 34 departamentos (nivel 2)
- **`dim_municipios`**: Los 1189 municipios (nivel 3) con su departamento ya resuelto

También creamos `dim_ambitos_citrep` para los 185 ámbitos específicos de la elección CITREP (Circunscripción Transitoria Especial de Paz).

In [9]:
# --- dim_departamentos: solo nivel 2 ---
dim_departamentos = dim_ambitos[dim_ambitos['nivel_id'] == 2][
    ['ambito_indice', 'ambito_nombre', 'ambito_codigo']
].rename(columns={
    'ambito_indice': 'depto_indice',
    'ambito_nombre': 'depto_nombre',
    'ambito_codigo': 'depto_codigo',
}).reset_index(drop=True)

print(f"dim_departamentos: {len(dim_departamentos)} filas")
dim_departamentos

dim_departamentos: 34 filas


,depto_indice,depto_nombre,depto_codigo
0,1,AMAZONAS,6000
1,2,ANTIOQUIA,0100
2,3,ARAUCA,4000
3,4,ATLANTICO,0300
4,5,BOGOTA D.C.,1600
5,6,BOLIVAR,0500
6,7,BOYACA,0700
7,8,CALDAS,0900
8,9,CAQUETA,4400
9,10,CASANARE,4600


In [10]:
# --- dim_municipios: solo nivel 3, con departamento resuelto ---
dim_municipios = dim_ambitos[dim_ambitos['nivel_id'] == 3][
    ['ambito_indice', 'ambito_nombre', 'ambito_codigo', 'departamento_nombre', 'departamento_codigo']
].rename(columns={
    'ambito_indice': 'mpio_indice',
    'ambito_nombre': 'mpio_nombre',
    'ambito_codigo': 'mpio_codigo',
    'departamento_nombre': 'depto_nombre',
    'departamento_codigo': 'depto_codigo',
}).reset_index(drop=True)

print(f"dim_municipios: {len(dim_municipios)} filas")
dim_municipios.head(10)

dim_municipios: 1189 filas


,mpio_indice,mpio_nombre,mpio_codigo,depto_nombre,depto_codigo
0,35,ABEJORRAL,0100004,ANTIOQUIA,0100
1,36,ABREGO,2500004,NORTE DE SAN,2500
2,37,ABRIAQUI,0100007,ANTIOQUIA,0100
3,38,ACACIAS,5200005,META,5200
4,39,ACANDI,1700004,CHOCO,1700
5,40,ACEVEDO,1900004,HUILA,1900
6,41,ACHI,0500004,BOLIVAR,0500
7,42,AGRADO,1900007,HUILA,1900
8,43,AGUA DE DIOS,1500004,CUNDINAMARCA,1500
9,44,AGUACHICA,1200075,CESAR,1200


In [11]:
# --- dim_ambitos_citrep: ámbitos específicos de CITREP (elección 7) ---
# CITREP = Circunscripción Transitoria Especial de Paz
ambitos_citrep = data['amb'][3]['ambitos']  # elec=7
lookup_citrep = {a['i']: a for a in ambitos_citrep}

filas_citrep = []
for a in ambitos_citrep:
    padre_indice = None
    if a['p']:
        padre_rel = a['p'][0]
        if padre_rel['p']:
            padre_indice = padre_rel['p'][0]
    
    fila = {
        'ambito_indice': a['i'],
        'ambito_nombre': a['n'],
        'ambito_codigo': a['co'],
        'nivel_id': a['l'],
        'padre_indice': padre_indice,
    }
    
    # Resolver departamento para municipios
    if a['l'] == 3 and padre_indice is not None:
        padre = lookup_citrep.get(padre_indice, {})
        fila['departamento_nombre'] = padre.get('n', '')
    elif a['l'] == 2:
        fila['departamento_nombre'] = a['n']
    else:
        fila['departamento_nombre'] = None
    
    filas_citrep.append(fila)

dim_ambitos_citrep = pd.DataFrame(filas_citrep)
print(f"dim_ambitos_citrep: {len(dim_ambitos_citrep)} filas")
print(f"  Departamentos CITREP: {len(dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 2])}")
print(f"  Municipios CITREP: {len(dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 3])}")
dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 3].head(10)

dim_ambitos_citrep: 185 filas
  Departamentos CITREP: 16
  Municipios CITREP: 168


,ambito_indice,ambito_nombre,ambito_codigo,nivel_id,padre_indice,departamento_nombre
17,17,ACANDI,1706004,3,13.0,CIRCUNSCRIPCIÓN 6 (CHOCÓ - ANT
18,18,AGUSTIN CODAZZI,1212150,3,4.0,CIRCUNSCRIPCIÓN 12 (CESAR - LA
19,19,ALBANIA,4405002,3,12.0,CIRCUNSCRIPCIÓN 5 (CAQUETÁ - H
20,20,ALGECIRAS,1905013,3,12.0,CIRCUNSCRIPCIÓN 5 (CAQUETÁ - H
21,21,AMALFI,0103016,3,10.0,CIRCUNSCRIPCIÓN 3 ANTIOQUIA
22,22,ANORI,0103028,3,10.0,CIRCUNSCRIPCIÓN 3 ANTIOQUIA
23,23,APARTADO,0116035,3,8.0,CIRCUNSCRIPCIÓN 16 ANTIOQUIA
24,24,ARACATACA,2112010,3,4.0,CIRCUNSCRIPCIÓN 12 (CESAR - LA
25,25,ARAUQUITA,4002010,3,9.0,CIRCUNSCRIPCIÓN 2 ARAUCA
26,26,ARENAL,0513005,3,5.0,CIRCUNSCRIPCIÓN 13 (BOLÍVAR -


## Paso 7: Resumen de tablas de dimensión creadas

Todas las tablas de dimensión están listas para ser usadas como lookup al procesar los archivos de resultados de votos.

| DataFrame | Filas | Descripción |
|-----------|-------|-------------|
| `dim_elecciones` | 7 | Elecciones + circunscripciones (Senado Nacional, Senado Indígena, Cámara Territorial, etc.) |
| `dim_niveles` | 7 | Jerarquía geográfica (Colombia → Depto → Municipio → Zona → Comuna → Puesto → Mesa) |
| `dim_partidos` | 348 | Partidos políticos con código, nombre y color |
| `dim_ambitos` | 1,224 | Todos los ámbitos geográficos (país + deptos + municipios) con jerarquía resuelta |
| `dim_departamentos` | 34 | Vista filtrada: solo departamentos |
| `dim_municipios` | 1,189 | Vista filtrada: solo municipios con su departamento |
| `dim_ambitos_citrep` | 185 | Ámbitos específicos de las Circunscripciones Transitorias Especiales de Paz |

**Uso**: Cuando los archivos de votos traigan un código como `"0100004"`, se puede buscar en `dim_ambitos` para saber que es **ABEJORRAL, ANTIOQUIA**. Igualmente, un `codpar = "1"` se traduce a **PARTIDO LIBERAL COLOMBIANO** usando `dim_partidos`.

In [12]:
# Resumen final: todas las tablas de dimensión en memoria
tablas = {
    'dim_elecciones': dim_elecciones,
    'dim_niveles': dim_niveles,
    'dim_partidos': dim_partidos,
    'dim_ambitos': dim_ambitos,
    'dim_departamentos': dim_departamentos,
    'dim_municipios': dim_municipios,
    'dim_ambitos_citrep': dim_ambitos_citrep,
}

print("Tablas de dimensión disponibles:")
print("-" * 50)
for nombre, df in tablas.items():
    print(f"  {nombre:25s} -> {len(df):>5} filas, {len(df.columns)} columnas")
print("-" * 50)
print(f"Total: {len(tablas)} tablas creadas")

Tablas de dimensión disponibles:
--------------------------------------------------
  dim_elecciones            ->     7 filas, 10 columnas
  dim_niveles               ->     7 filas, 2 columnas
  dim_partidos              ->   348 filas, 5 columnas
  dim_ambitos               ->  1224 filas, 8 columnas
  dim_departamentos         ->    34 filas, 3 columnas
  dim_municipios            ->  1189 filas, 5 columnas
  dim_ambitos_citrep        ->   185 filas, 6 columnas
--------------------------------------------------
Total: 7 tablas creadas


## Datos granulares: resultados de la elección de Senado (SE)

La elección de Senado tiene **2 circunscripciones** (cámaras):
- **Nacional** (cam_id=0): 100 curules por circunscripción nacional
- **Indígenas** (cam_id=4): circunscripción especial indígena (2 curules)

Cada response del API contiene un array `camaras` con **2 entradas** que debemos iterar.

Primero exploramos la estructura de la respuesta del API para entender los datos disponibles.

In [13]:
# Explorar estructura de la respuesta de Senado (SE) a nivel nacional
url_se = "https://resultados.registraduria.gov.co/json/ACT/SE/00.json"
r_se = requests.get(url_se, cookies=cookies, headers=headers)
se_data = r_se.json()

print(f"Status: {r_se.status_code}")
print(f"Llaves top: {list(se_data.keys())}")
print(f"Número de cámaras: {len(se_data['camaras'])}")

for i, cam in enumerate(se_data['camaras']):
    print(f"\n{'='*60}")
    print(f"--- Cámara {i} ---")
    print(f"  Llaves: {list(cam.keys())}")
    print(f"  cam: {cam['cam']}, cir: {cam['cir']}")
    n_part = len(cam.get('partotabla', []))
    n_mapa = len(cam.get('mapagan', []))
    print(f"  partotabla: {n_part} partidos")
    print(f"  mapagan: {n_mapa} sub-ámbitos")
    
    # Totales
    if 'totales' in cam:
        t = cam['totales'].get('act', {})
        print(f"  Totales: votant={t.get('votant')}, votval={t.get('votval')}, votnul={t.get('votnul')}")
    
    # Ejemplo de un partido
    if cam.get('partotabla'):
        p0 = cam['partotabla'][0]['act']
        print(f"  Ejemplo partido: codpar={p0['codpar']}, vot={p0['vot']}")
        print(f"    cantotabla: {len(p0.get('cantotabla', []))} candidatos")
        if p0.get('cantotabla'):
            c0 = p0['cantotabla'][0]
            print(f"    Ejemplo candidato: {c0.get('nomcan','')} {c0.get('apecan','')}, vot={c0.get('vot','')}")

Status: 200
Llaves top: ['elec', 'amb', 'tope', 'numact', 'numdep', 'iscircus', 'mdhm', 'shc', 'totales', 'camaras', 'historico', 'dept']
Número de cámaras: 2

--- Cámara 0 ---
  Llaves: ['cam', 'cir', 'carg', 'cargElectos', 'cargEmpatados', 'cargtota', 'totales', 'partotabla', 'mapagan']
  cam: 0, cir: 1
  partotabla: 16 partidos
  mapagan: 34 sub-ámbitos
  Totales: votant=20492278, votval=19423187, votnul=573572
  Ejemplo partido: codpar=92, vot=4413636
    cantotabla: 96 candidatos
    Ejemplo candidato: SOLO POR LA LISTA , vot=4413636

--- Cámara 1 ---
  Llaves: ['cam', 'cir', 'carg', 'cargElectos', 'cargEmpatados', 'cargtota', 'totales', 'partotabla', 'mapagan']
  cam: 4, cir: 1
  partotabla: 10 partidos
  mapagan: 34 sub-ámbitos
  Totales: votant=408336, votval=303979, votnul=49967
  Ejemplo partido: codpar=11, vot=88294
    cantotabla: 4 candidatos
    Ejemplo candidato: MARTHA ISABEL PERALTA EPIEYU, vot=61098


In [14]:
# Explorar respuesta a nivel departamento (ej: Antioquia)
# Primero probamos con el código de departamento de dim_departamentos
antioquia_code = dim_departamentos[dim_departamentos['depto_nombre'] == 'ANTIOQUIA']['depto_codigo'].values[0]
url_depto = f"https://resultados.registraduria.gov.co/json/ACT/SE/{antioquia_code}.json"
r_depto = requests.get(url_depto, cookies=cookies, headers=headers)
depto_data = r_depto.json()
print(f"Respuesta para departamento Antioquia ({antioquia_code}):")
print(f"Número de cámaras: {len(depto_data['camaras'])}")
for i, cam in enumerate(depto_data['camaras']):
    n_part = len(cam.get('partotabla', []))
    n_mapa = len(cam.get('mapagan', []))
    n_cand_total = sum(len(p['act'].get('cantotabla', [])) for p in cam.get('partotabla', []))
    print(f"  Cámara {i}: cam={cam['cam']}, cir={cam['cir']}, partidos={n_part}, candidatos={n_cand_total}, mapagan={n_mapa}")

# Explorar respuesta a nivel municipio (ej: Medellín con código 7 dígitos)
medellin = dim_ambitos[(dim_ambitos['nivel_id'] == 3) & (dim_ambitos['ambito_nombre'] == 'MEDELLIN')]
medellin_code = medellin['ambito_codigo'].values[0]
url_mpio = f"https://resultados.registraduria.gov.co/json/ACT/SE/{medellin_code}.json"
r_mpio = requests.get(url_mpio, cookies=cookies, headers=headers)
print(f"\n{'='*60}")
print(f"Respuesta para municipio Medellín ({medellin_code}):")
print(f"  Status: {r_mpio.status_code}")
if r_mpio.status_code == 200:
    mpio_data = r_mpio.json()
    print(f"  Número de cámaras: {len(mpio_data['camaras'])}")
    for i, cam in enumerate(mpio_data['camaras']):
        n_part = len(cam.get('partotabla', []))
        n_cand_total = sum(len(p['act'].get('cantotabla', [])) for p in cam.get('partotabla', []))
        t = cam['totales'].get('act', {})
        print(f"  Cámara {i}: cam={cam['cam']}, partidos={n_part}, candidatos={n_cand_total}")
        print(f"    totales: votant={t.get('votant')}, votval={t.get('votval')}")
else:
    print(f"  Respuesta vacía o error - primeros 200 chars: {r_mpio.text[:200]}")

Respuesta para departamento Antioquia (0100):
Número de cámaras: 2
  Cámara 0: cam=0, cir=0, partidos=16, candidatos=1064, mapagan=125
  Cámara 1: cam=4, cir=0, partidos=10, candidatos=33, mapagan=125

Respuesta para municipio Medellín (0100001):
  Status: 200
  Número de cámaras: 2
  Cámara 0: cam=0, partidos=16, candidatos=1064
    totales: votant=913680, votval=879366
  Cámara 1: cam=4, partidos=10, candidatos=33
    totales: votant=5168, votval=3444


In [ ]:
# Explorar mapagan (municipios) y partotabla (candidatos) a nivel departamento
cam_nac = depto_data['camaras'][0]  # Nacional en Antioquia

print("=== MAPAGAN (municipios ganador) ===")
print(f"Total: {len(cam_nac['mapagan'])} sub-ámbitos")
if cam_nac['mapagan']:
    m0 = cam_nac['mapagan'][0]
    print(f"Llaves: {list(m0.keys())}")
    print(f"Ejemplo: {m0}")
    print("\nPrimeros 5 municipios:")
    for m in cam_nac['mapagan'][:5]:
        print(f"  {m.get('amb')} - {m.get('nombre','?')}: ganador codpar={m.get('codpar')}, vot={m.get('vot')}, votant={m.get('votant')}")

print("\n=== PARTOTABLA (candidatos por partido) ===")
print(f"Total partidos: {len(cam_nac['partotabla'])}")
if cam_nac['partotabla']:
    p0 = cam_nac['partotabla'][0]['act']
    print(f"Llaves partido: {list(p0.keys())}")
    print(f"Partido: codpar={p0['codpar']}, vot={p0['vot']}")
    print(f"Candidatos: {len(p0.get('cantotabla', []))}")
    if p0.get('cantotabla'):
        c0 = p0['cantotabla'][0]
        print(f"  Llaves candidato: {list(c0.keys())}")
        print(f"  Ejemplo: {c0}")

# Verificar si mapagan tiene datos por cámara diferente
print("\n=== COMPARAR MAPAGAN POR CÁMARA ===")
for i, cam in enumerate(depto_data['camaras']):
    print(f"Cámara {i} (cam={cam['cam']}): mapagan={len(cam['mapagan'])} items")
    if cam['mapagan']:
        print(f"  Ejemplo: amb={cam['mapagan'][0]['amb']}, codpar={cam['mapagan'][0].get('codpar')}, vot={cam['mapagan'][0].get('vot')}")

=== MAPAGAN (municipios ganador) ===
Total: 125 sub-ámbitos
Llaves: ['amb', 'codpar', 'vot', 'pvot', 'votant', 'pvotant', 'votcan', 'mesesc', 'pmesesc', 'carg', 'nombre', 'hayEmpate']
Ejemplo: {'amb': '0100004', 'codpar': '10', 'vot': '2555', 'pvot': '48,83%', 'votant': '5873', 'pvotant': '37,54%', 'votcan': '5095', 'mesesc': '51', 'pmesesc': '100%', 'carg': '0', 'nombre': 'ABEJORRAL', 'hayEmpate': '0'}

Primeros 5 municipios:
  0100004 - ABEJORRAL: ganador codpar=10, vot=2555, votant=5873
  0100007 - ABRIAQUI: ganador codpar=3, vot=793, votant=1089
  0100010 - ALEJANDRIA: ganador codpar=2, vot=548, votant=2259
  0100013 - AMAGA: ganador codpar=10, vot=1822, votant=9675
  0100016 - AMALFI: ganador codpar=10, vot=871, votant=4482

=== PARTOTABLA (candidatos por partido) ===
Total partidos: 16
Llaves partido: ['codpar', 'cam', 'vot', 'pvot', 'carg', 'cargElectos', 'cargEmpatados', 'cantotabla']
Partido: codpar=10, vot=744372
Candidatos: 65
  Llaves candidato: ['amb', 'codcan', 'cedula', 

### Hallazgos de exploración

| Nivel | Endpoint | Resultado |
|-------|----------|-----------|
| Nacional | `/json/ACT/SE/00.json` | 2 cámaras (Nacional + Indígenas), candidatos en `partotabla`, departamentos ganador en `mapagan` |
| Departamento | `/json/ACT/SE/{depto_codigo}.json` | 2 cámaras, candidatos en `partotabla`, municipios ganador en `mapagan` |
| Municipio | `/json/ACT/SE/{ambito_7dig}.json` | ✅ 2 cámaras, candidatos en `partotabla` — requiere códigos de 7 dígitos de `dim_ambitos` (nivel_id=3) |

**Máxima granularidad posible**: nivel municipio, con 2 circunscripciones por response.
> **Nota**: Los códigos de 5 dígitos DANE devuelven 404. Se deben usar los códigos de 7 dígitos del nomenclátor (`dim_ambitos.ambito_codigo` donde `nivel_id == 3`).

### Estructura de cada response

Cada response contiene `camaras[2]`, una por circunscripción:

| `cam` | Circunscripción | Descripción |
|-------|----------------|-------------|
| `0` | Nacional | 100 curules — candidatos de todo el país |
| `4` | Indígenas | 2 curules — circunscripción especial indígena |

Cada cámara contiene:
- **`partotabla`**: Partidos → candidatos con votos (la tabla más granular)
- **`mapagan`**: Resumen de ganador por municipio dentro del departamento (solo a nivel depto)
- **`totales`**: Votantes, votos válidos, nulos, no marcados

### Estrategia de scraping

**Nivel departamento** — 34 requests (uno por departamento) → **2 tablas**:
1. **`se_candidatos_depto`**: Votos de cada candidato por partido, departamento y circunscripción
2. **`se_municipios_ganador`**: Partido ganador y votos por municipio y circunscripción (desde `mapagan`)

**Nivel municipio** — 1,189 requests en paralelo (20 workers) → **1 tabla**:
3. **`se_candidatos_municipio`**: Votos de cada candidato por partido, municipio y circunscripción (máxima granularidad)

In [16]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
import threading

# === Scraping paralelo de los 34 departamentos para Senado (SE) ===
BASE_URL = "https://resultados.registraduria.gov.co/json/ACT/SE"
MAX_WORKERS = 10

# Lookup de circunscripciones
CIRCUNSCRIPCIONES = {
    '0': 'Nacional',
    '4': 'Indígenas',
}

# Session con connection pool
session = requests.Session()
session.cookies.update(cookies)
session.headers.update(headers)
adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
session.mount('https://', adapter)

lock = threading.Lock()
completados = 0

def fetch_departamento(row):
    """Descarga y parsea los resultados de Senado de un departamento."""
    global completados
    codigo = row['depto_codigo']
    nombre = row['depto_nombre']
    
    r = session.get(f"{BASE_URL}/{codigo}.json")
    
    with lock:
        completados += 1
        print(f"  [{completados:2d}/34] {'✅' if r.status_code == 200 else '❌'} {nombre} ({codigo})")
    
    if r.status_code != 200:
        return None, (codigo, nombre, r.status_code)
    
    return r.json(), None

# Ejecutar en paralelo
rows = [row for _, row in dim_departamentos.iterrows()]
responses_se = {}
errores = []

print(f"Descargando datos de {len(rows)} departamentos ({MAX_WORKERS} workers)...")
t0 = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_departamento, row): row for row in rows}
    for future in as_completed(futures):
        row = futures[future]
        resp, error = future.result()
        if resp:
            responses_se[row['depto_codigo']] = resp
        if error:
            errores.append(error)

elapsed = time.time() - t0
print(f"\n✅ Completado en {elapsed:.1f}s: {len(responses_se)} OK, {len(errores)} errores")
if errores:
    print(f"   Errores: {errores}")

Descargando datos de 34 departamentos (10 workers)...
  [ 1/34] ✅ BOYACA (0700)
  [ 2/34] ✅ BOLIVAR (0500)
  [ 3/34] ✅ ATLANTICO (0300)
  [ 4/34] ✅ AMAZONAS (6000)
  [ 5/34] ✅ ANTIOQUIA (0100)
  [ 6/34] ✅ CESAR (1200)
  [ 7/34] ✅ CONSULADOS (8800)
  [ 8/34] ✅ CORDOBA (1300)
  [ 9/34] ✅ CAUCA (1100)
  [10/34] ✅ ARAUCA (4000)
  [11/34] ✅ BOGOTA D.C. (1600)
  [12/34] ✅ CASANARE (4600)
  [13/34] ✅ CALDAS (0900)
  [14/34] ✅ LA GUAJIRA (4800)
  [15/34] ✅ CHOCO (1700)
  [16/34] ✅ NARIÑO (2300)
  [17/34] ✅ CUNDINAMARCA (1500)
  [18/34] ✅ CAQUETA (4400)
  [19/34] ✅ NORTE DE SAN (2500)
  [20/34] ✅ HUILA (1900)
  [21/34] ✅ GUAVIARE (5400)
  [22/34] ✅ META (5200)
  [23/34] ✅ GUAINIA (5000)
  [24/34] ✅ QUINDIO (2600)
  [25/34] ✅ MAGDALENA (2100)
  [26/34] ✅ VALLE (3100)
  [27/34] ✅ PUTUMAYO (6400)
  [28/34] ✅ VICHADA (7200)
  [29/34] ✅ RISARALDA (2400)
  [30/34] ✅ VAUPES (6800)
  [31/34] ✅ SAN ANDRES (5600)
  [32/34] ✅ SANTANDER (2700)
  [33/34] ✅ SUCRE (2800)
  [34/34] ✅ TOLIMA (2900)

✅ Completad

### Construyendo DataFrames estructurados

De cada response departamental extraemos dos tablas (por cada circunscripción):

1. **`se_candidatos_depto`**: Votos de cada candidato por partido, departamento y circunscripción (desde `partotabla`)
2. **`se_municipios_ganador`**: Partido ganador y votos por municipio y circunscripción (desde `mapagan`)

In [17]:
# === Tabla 1: se_candidatos_depto ===
filas_candidatos = []

for depto_codigo, resp in responses_se.items():
    depto_nombre = dim_departamentos[
        dim_departamentos['depto_codigo'] == depto_codigo
    ]['depto_nombre'].values[0]
    
    for cam in resp['camaras']:
        cam_id = cam['cam']
        circunscripcion = CIRCUNSCRIPCIONES.get(cam_id, f'Desconocida ({cam_id})')
        
        for partido in cam.get('partotabla', []):
            p = partido['act']
            # Convertimos a str aquí mismo para asegurar consistencia
            codpar = str(p['codpar']) 
            votos_partido = p['vot']
            
            for candidato in p.get('cantotabla', []):
                filas_candidatos.append({
                    'depto_codigo': depto_codigo,
                    'depto_nombre': depto_nombre,
                    'circunscripcion_id': cam_id,
                    'circunscripcion': circunscripcion,
                    'partido_codigo': codpar,
                    'votos_partido': int(votos_partido),
                    'candidato_codigo': candidato['codcan'],
                    'candidato_cedula': candidato['cedula'],
                    'candidato_nombre': f"{candidato['nomcan']} {candidato['apecan']}".strip(),
                    'candidato_votos': int(candidato['vot']),
                    'candidato_porcentaje': candidato['pvot'],
                    'preferido': candidato.get('pref', '0'),
                })

se_candidatos_depto = pd.DataFrame(filas_candidatos)

# --- AJUSTE DEL MERGE ---
_lookup_partido = dim_partidos[['partido_indice', 'partido_nombre']].copy()
_lookup_partido['partido_indice'] = _lookup_partido['partido_indice'].astype(str)

se_candidatos_depto = se_candidatos_depto.merge(
    _lookup_partido,
    left_on='partido_codigo',
    right_on='partido_indice',
    how='left'
).drop(columns=['partido_indice'])

# Opcional: Renombrar para que quede igual a las otras tablas si lo prefieres
# se_candidatos_depto = se_candidatos_depto.rename(columns={'partido_nombre': 'nombre_del_partido'})

print(f"se_candidatos_depto: {len(se_candidatos_depto)} filas")
print(f"  Departamentos: {se_candidatos_depto['depto_nombre'].nunique()}")
print(f"  Circunscripciones: {se_candidatos_depto['circunscripcion'].nunique()}")
print(f"  Candidatos únicos: {se_candidatos_depto['candidato_nombre'].nunique()}")
print(f"  Partidos: {se_candidatos_depto['partido_nombre'].nunique()}")

# Validación preventiva de nulos
nulos = se_candidatos_depto['partido_nombre'].isna().sum()
if nulos > 0:
    print(f"⚠️ Alerta: {nulos} registros no encontraron nombre de partido.")

print(f"\nFilas por circunscripción:")
print(se_candidatos_depto.groupby('circunscripcion').size())
se_candidatos_depto.head(10)

se_candidatos_depto: 37298 filas
  Departamentos: 34
  Circunscripciones: 2
  Candidatos únicos: 1072
  Partidos: 26

Filas por circunscripción:
circunscripcion
Indígenas     1122
Nacional     36176
dtype: int64


,depto_codigo,depto_nombre,circunscripcion_id,circunscripcion,partido_codigo,votos_partido,candidato_codigo,candidato_cedula,candidato_nombre,candidato_votos,candidato_porcentaje,preferido,partido_nombre
0,0700,BOYACA,0,Nacional,57,119757,5,1053664038,JOHN EDICKSON AMAYA RODRIGUEZ,61430,"12,20%",1,ALIANZA POR COLOMBIA
1,0700,BOYACA,0,Nacional,57,119757,3,80738375,ARIEL FERNANDO AVILA MARTINEZ,10315,"2,04%",1,ALIANZA POR COLOMBIA
2,0700,BOYACA,0,Nacional,57,119757,0,,SOLO POR LA LISTA,9554,"1,89%",1,ALIANZA POR COLOMBIA
3,0700,BOYACA,0,Nacional,57,119757,100,1095927582,JONATHAN FERNEY PULIDO HERNANDEZ,6393,"1,26%",1,ALIANZA POR COLOMBIA
4,0700,BOYACA,0,Nacional,57,119757,20,1035857000,GUSTAVO ADOLFO MORENO HURTADO,6255,"1,24%",1,ALIANZA POR COLOMBIA
5,0700,BOYACA,0,Nacional,57,119757,39,1088299925,LUIS CARLOS RUA SANCHEZ,2992,"0,59%",1,ALIANZA POR COLOMBIA
6,0700,BOYACA,0,Nacional,57,119757,14,52440836,ANDREA PADILLA VILLARRAGA,2855,"0,56%",1,ALIANZA POR COLOMBIA
7,0700,BOYACA,0,Nacional,57,119757,6,1098616391,JUAN CARLOS BOCANEGRA CHACON,2591,"0,51%",1,ALIANZA POR COLOMBIA
8,0700,BOYACA,0,Nacional,57,119757,22,88305820,JAIRO ALBERTO CASTELLANOS SERRANO,1233,"0,24%",1,ALIANZA POR COLOMBIA
9,0700,BOYACA,0,Nacional,57,119757,2,1030522325,LUVI KATHERINE MIRANDA PEÑA,1216,"0,24%",1,ALIANZA POR COLOMBIA


In [18]:
# === Tabla 2: se_municipios_ganador ===
# Partido ganador y votos por municipio y circunscripción (desde mapagan)
filas_municipios = []

for depto_codigo, resp in responses_se.items():
    depto_nombre = dim_departamentos[
        dim_departamentos['depto_codigo'] == depto_codigo
    ]['depto_nombre'].values[0]
    
    for cam in resp['camaras']:
        cam_id = cam['cam']
        circunscripcion = CIRCUNSCRIPCIONES.get(cam_id, f'Desconocida ({cam_id})')
        
        for mpio in cam.get('mapagan', []):
            filas_municipios.append({
                'depto_codigo': depto_codigo,
                'depto_nombre': depto_nombre,
                'circunscripcion_id': cam_id,
                'circunscripcion': circunscripcion,
                'mpio_codigo': mpio['amb'],
                'mpio_nombre': mpio.get('nombre', ''),
                'ganador_partido_codigo': mpio['codpar'],
                'ganador_votos': int(mpio['vot']),
                'ganador_porcentaje': mpio['pvot'],
                'total_votantes': int(mpio['votant']),
                'participacion': mpio['pvotant'],
                'votos_validos': int(mpio['votcan']),
                'mesas_escrutadas': int(mpio['mesesc']),
                'pct_mesas_escrutadas': mpio['pmesesc'],
                'hay_empate': mpio.get('hayEmpate', '0'),
            })

se_municipios_ganador = pd.DataFrame(filas_municipios)

# Enriquecer con nombre del partido ganador
# Nota: el codpar del API corresponde a partido_indice del nomenclátor, NO a partido_codigo
_lookup_partido = dim_partidos[['partido_indice', 'partido_nombre']].copy()
_lookup_partido['partido_indice'] = _lookup_partido['partido_indice'].astype(str)
se_municipios_ganador = se_municipios_ganador.merge(
    _lookup_partido,
    left_on='ganador_partido_codigo',
    right_on='partido_indice',
    how='left'
).drop(columns=['partido_indice']).rename(columns={'partido_nombre': 'ganador_partido_nombre'})

print(f"se_municipios_ganador: {len(se_municipios_ganador)} filas")
print(f"  Departamentos: {se_municipios_ganador['depto_nombre'].nunique()}")
print(f"  Municipios: {se_municipios_ganador['mpio_nombre'].nunique()}")
print(f"  Circunscripciones: {se_municipios_ganador['circunscripcion'].nunique()}")
print("\nFilas por circunscripción:")
print(se_municipios_ganador.groupby('circunscripcion').size())
print("\nGanador por circunscripción (Top 3 partidos):")
for circ in se_municipios_ganador['circunscripcion'].unique():
    subset = se_municipios_ganador[se_municipios_ganador['circunscripcion'] == circ]
    print(f"\n  {circ}:")
    print(subset['ganador_partido_nombre'].value_counts().head(3).to_string(header=False))
se_municipios_ganador.head(10)

se_municipios_ganador: 2378 filas
  Departamentos: 34
  Municipios: 1100
  Circunscripciones: 2

Filas por circunscripción:
circunscripcion
Indígenas    1189
Nacional     1189
dtype: int64

Ganador por circunscripción (Top 3 partidos):

  Nacional:
PARTIDO CENTRO DEMOCRÁTICO    257
PACTO HISTÓRICO SENADO        227
PARTIDO LIBERAL COLOMBIANO    187

  Indígenas:
MOVIMIENTO AUTORIDADES INDÍGENAS DE COLOMBIA "AICO"    515
MOVIMIENTO ALTERNATIVO INDÍGENA Y SOCIAL "MAIS"        461
MOVIMIENTO UNIDAD EN MINGA POR COLOMBIA                111


,depto_codigo,depto_nombre,circunscripcion_id,circunscripcion,mpio_codigo,mpio_nombre,ganador_partido_codigo,ganador_votos,ganador_porcentaje,total_votantes,participacion,votos_validos,mesas_escrutadas,pct_mesas_escrutadas,hay_empate,ganador_partido_nombre
0,0700,BOYACA,0,Nacional,0700007,ALMEIDA,57,216,"30,29%",783,"38,87%",696,7,100%,0,ALIANZA POR COLOMBIA
1,0700,BOYACA,0,Nacional,0700008,AQUITANIA (PUEBLOVIEJO),10,1195,"25,12%",5211,"39,40%",4574,47,100%,0,PARTIDO CENTRO DEMOCRÁTICO
2,0700,BOYACA,0,Nacional,0700010,ARCABUCO,57,617,"24,92%",2645,"50,34%",2389,16,100%,0,ALIANZA POR COLOMBIA
3,0700,BOYACA,0,Nacional,0700013,BELEN,57,940,"27,38%",3638,"51,31%",3320,22,100%,0,ALIANZA POR COLOMBIA
4,0700,BOYACA,0,Nacional,0700016,BERBEO,57,265,"37,32%",772,"51,15%",679,5,100%,0,ALIANZA POR COLOMBIA
5,0700,BOYACA,0,Nacional,0700019,BETEITIVA,57,284,"30,50%",1002,"50,70%",911,7,100%,0,ALIANZA POR COLOMBIA
6,0700,BOYACA,0,Nacional,0700022,BOAVITA,10,548,"27,69%",2098,"41,56%",1941,17,100%,0,PARTIDO CENTRO DEMOCRÁTICO
7,0700,BOYACA,0,Nacional,0700025,BOYACA,57,645,"38,66%",1820,"39,56%",1631,14,100%,0,ALIANZA POR COLOMBIA
8,0700,BOYACA,0,Nacional,0700028,BRICEÑO,57,516,"50,09%",1115,"44,65%",1018,8,100%,0,ALIANZA POR COLOMBIA
9,0700,BOYACA,0,Nacional,0700031,BUENAVISTA,57,509,"23,38%",2387,"46,99%",2138,17,100%,0,ALIANZA POR COLOMBIA


In [19]:
# === Resumen: votos totales por candidato a nivel nacional (por circunscripción) ===
for circ in ['Nacional', 'Indígenas']:
    subset = se_candidatos_depto[se_candidatos_depto['circunscripcion'] == circ]
    resumen = subset.groupby(['partido_nombre', 'candidato_nombre']).agg(
        votos_totales=('candidato_votos', 'sum'),
        deptos_presencia=('depto_codigo', 'nunique'),
    ).sort_values('votos_totales', ascending=False).reset_index()
    
    print(f"\n{'='*80}")
    print(f"Top 10 candidatos - {circ} ({len(resumen)} candidatos, {subset['candidato_votos'].sum():,} votos totales)")
    print(f"{'='*80}")
    for _, row in resumen.head(10).iterrows():
        print(f"  {row['votos_totales']:>10,}  {row['candidato_nombre'][:40]:<42s} ({row['partido_nombre'][:30]})")


Top 10 candidatos - Nacional (1064 candidatos, 18,806,189 votos totales)
   4,413,636  SOLO POR LA LISTA                          (PACTO HISTÓRICO SENADO)
   3,035,715  SOLO POR LA LISTA                          (PARTIDO CENTRO DEMOCRÁTICO)
     216,376  SOLO POR LA LISTA                          (PARTIDO LIBERAL COLOMBIANO)
     182,596  SOLO POR LA LISTA                          (PARTIDO CONSERVADOR COLOMBIANO)
     178,907  NADYA GEORGETTE BLEL SCAFF                 (PARTIDO CONSERVADOR COLOMBIANO)
     175,388  LIDIO ARTURO GARCIA TURBAY                 (PARTIDO LIBERAL COLOMBIANO)
     166,070  SOLO POR LA LISTA                          (ALIANZA POR COLOMBIA)
     159,956  JONATHAN FERNEY PULIDO HERNANDEZ           (ALIANZA POR COLOMBIA)
     146,563  SOLO POR LA LISTA                          (PARTIDO DE LA UNIÓN POR LA GEN)
     146,507  YESSID ENRIQUE PULGAR DAZA                 (PARTIDO LIBERAL COLOMBIANO)

Top 10 candidatos - Indígenas (33 candidatos, 245,440 votos totales)


## Exportación de datos

Disponibilizamos los 2 DataFrames en múltiples formatos dentro de `data/raw_data_extract/senado/`:

| DataFrame | Descripción |
|-----------|-------------|
| `se_candidatos_depto` | Votos de cada candidato por partido, departamento y circunscripción |
| `se_municipios_ganador` | Partido ganador por municipio y circunscripción |

| Formato | Archivo | Uso típico |
|---------|---------|------------|
| CSV (coma) | `.csv` | Compatible universal, Excel, R, Python |
| CSV (punto y coma) | `_semicolon.csv` | Excel en configuraciones regionales con coma decimal |
| Excel | `.xlsx` | Análisis rápido en hojas de cálculo |
| JSON | `.json` | APIs, JavaScript, visualizaciones web |
| Parquet | `.parquet` | Alto rendimiento para grandes volúmenes, Spark, pandas |

In [20]:
import os

# Directorio de salida (relativo a la raíz del proyecto)
OUTPUT_DIR = os.path.join('..', '..', 'data', 'raw_data_extract', 'senado')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Exportar ambos DataFrames
datasets = {
    'se_candidatos_depto': se_candidatos_depto,
    'se_municipios_ganador': se_municipios_ganador,
}

for nombre, df in datasets.items():
    base = os.path.join(OUTPUT_DIR, nombre)
    
    # CSV (separado por coma)
    df.to_csv(f'{base}.csv', index=False, encoding='utf-8-sig')
    
    # CSV (separado por punto y coma)
    df.to_csv(f'{base}_semicolon.csv', index=False, encoding='utf-8-sig', sep=';')
    
    # Excel
    df.to_excel(f'{base}.xlsx', index=False, engine='openpyxl')
    
    # JSON
    df.to_json(f'{base}.json', orient='records', force_ascii=False, indent=2)
    
    # Parquet
    df.to_parquet(f'{base}.parquet', index=False, engine='pyarrow')
    
    print(f"\n✅ {nombre} ({len(df)} filas) — 5 archivos exportados:")
    for ext in ['.csv', '_semicolon.csv', '.xlsx', '.json', '.parquet']:
        filepath = f'{base}{ext}'
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"   {os.path.basename(filepath):45s} {size_mb:.2f} MB")


✅ se_candidatos_depto (37298 filas) — 5 archivos exportados:
   se_candidatos_depto.csv                       3.97 MB
   se_candidatos_depto_semicolon.csv             3.92 MB
   se_candidatos_depto.xlsx                      2.52 MB
   se_candidatos_depto.json                      15.45 MB
   se_candidatos_depto.parquet                   0.23 MB

✅ se_municipios_ganador (2378 filas) — 5 archivos exportados:
   se_municipios_ganador.csv                     0.29 MB
   se_municipios_ganador_semicolon.csv           0.28 MB
   se_municipios_ganador.xlsx                    0.19 MB
   se_municipios_ganador.json                    1.18 MB
   se_municipios_ganador.parquet                 0.08 MB


## Scraping a nivel municipal: votos por candidato en cada municipio

**Hallazgo clave**: Los endpoints de 7 dígitos (`/json/ACT/SE/{ambito_codigo}.json`) SÍ devuelven datos de candidatos a nivel municipal — con las 2 circunscripciones. Los códigos de 7 dígitos corresponden al `ambito_codigo` de `dim_ambitos` (nivel 3 = municipio).

**Estrategia**:
1. Consultar los **1,189 municipios** individualmente con `ThreadPoolExecutor` + `requests.Session`
2. De cada response extraer `partotabla` → candidatos de las 2 circunscripciones
3. Construir `se_candidatos_municipio`: la tabla más granular posible

In [21]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
import threading

# === Scraping paralelo de los 1,189 municipios para Senado (SE) ===
BASE_URL = "https://resultados.registraduria.gov.co/json/ACT/SE"
MAX_WORKERS = 20

CIRCUNSCRIPCIONES = {
    '0': 'Nacional',
    '4': 'Indígenas',
}

# Municipios: nivel_id == 3 en dim_ambitos (código de 7 dígitos)
municipios = dim_ambitos[dim_ambitos['nivel_id'] == 3].copy()
total = len(municipios)
print(f"Municipios a consultar: {total}")

# Session con connection pool
session = requests.Session()
session.cookies.update(cookies)
session.headers.update(headers)
adapter = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
session.mount('https://', adapter)

lock = threading.Lock()
completados = 0

def fetch_municipio_se(row):
    """Descarga y parsea los candidatos de Senado de un municipio."""
    global completados
    codigo = row['ambito_codigo']
    nombre = row['ambito_nombre']
    depto = row['departamento_nombre']
    
    r = session.get(f"{BASE_URL}/{codigo}.json")
    
    with lock:
        completados += 1
        if completados % 200 == 0:
            print(f"  [{completados:4d}/{total}] procesados...")
    
    if r.status_code != 200:
        return [], (codigo, nombre, r.status_code)
    
    resp = r.json()
    filas = []
    
    for cam in resp.get('camaras', []):
        cam_id = cam['cam']
        circunscripcion = CIRCUNSCRIPCIONES.get(cam_id, f'Desconocida ({cam_id})')
        totales = cam.get('totales', {}).get('act', {})
        
        for partido in cam.get('partotabla', []):
            p = partido['act']
            for candidato in p.get('cantotabla', []):
                filas.append({
                    'depto_nombre': depto,
                    'mpio_codigo': codigo,
                    'mpio_nombre': nombre,
                    'circunscripcion_id': cam_id,
                    'circunscripcion': circunscripcion,
                    'partido_codigo': p['codpar'],
                    'votos_partido_mpio': int(p['vot']),
                    'candidato_codigo': candidato['codcan'],
                    'candidato_cedula': candidato['cedula'],
                    'candidato_nombre': f"{candidato['nomcan']} {candidato['apecan']}".strip(),
                    'candidato_votos': int(candidato['vot']),
                    'candidato_porcentaje': candidato['pvot'],
                    'preferido': candidato.get('pref', '0'),
                    'mpio_votantes': int(totales.get('votant', 0)),
                    'mpio_votos_validos': int(totales.get('votval', 0)),
                    'mpio_votos_nulos': int(totales.get('votnul', 0)),
                    'mpio_votos_no_marcados': int(totales.get('votnma', 0)),
                })
    return filas, None

rows = [row for _, row in municipios.iterrows()]
filas_se_mpio = []
errores_mpio = []

print(f"Descargando datos de {total} municipios ({MAX_WORKERS} workers concurrentes)...")
t0 = time.time()
completados = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_municipio_se, row): row for row in rows}
    for future in as_completed(futures):
        filas, error = future.result()
        filas_se_mpio.extend(filas)
        if error:
            errores_mpio.append(error)

elapsed = time.time() - t0
print(f"\n✅ Completado en {elapsed:.1f}s: {total - len(errores_mpio)} OK, {len(errores_mpio)} errores")
print(f"   Filas extraídas: {len(filas_se_mpio):,}")
if errores_mpio:
    print(f"   Errores: {errores_mpio[:5]}")

Municipios a consultar: 1189
Descargando datos de 1189 municipios (20 workers concurrentes)...
  [ 200/1189] procesados...
  [ 400/1189] procesados...
  [ 600/1189] procesados...
  [ 800/1189] procesados...
  [1000/1189] procesados...

✅ Completado en 22.9s: 1189 OK, 0 errores
   Filas extraídas: 1,304,333


In [22]:
# === DataFrame final: se_candidatos_municipio ===
se_candidatos_municipio = pd.DataFrame(filas_se_mpio)

# 1. Asegurar que partido_codigo sea string para evitar fallos de formato
se_candidatos_municipio['partido_codigo'] = se_candidatos_municipio['partido_codigo'].astype(str)

# 2. Preparar el lookup de partidos asegurando el tipo string
_lookup_partido = dim_partidos[['partido_indice', 'partido_nombre']].copy()
_lookup_partido['partido_indice'] = _lookup_partido['partido_indice'].astype(str)

# 3. Realizar el merge (ahora garantizado entre Strings)
se_candidatos_municipio = se_candidatos_municipio.merge(
    _lookup_partido,
    left_on='partido_codigo',
    right_on='partido_indice',
    how='left'
).drop(columns=['partido_indice'])

# --- Reportes y Verificación ---
print(f"se_candidatos_municipio: {len(se_candidatos_municipio):,} filas × {len(se_candidatos_municipio.columns)} columnas")
print(f"  Departamentos: {se_candidatos_municipio['depto_nombre'].nunique()}")
print(f"  Municipios: {se_candidatos_municipio['mpio_nombre'].nunique()}")
print(f"  Circunscripciones: {se_candidatos_municipio['circunscripcion'].nunique()}")
print(f"  Candidatos únicos: {se_candidatos_municipio['candidato_nombre'].nunique()}")
print(f"  Partidos: {se_candidatos_municipio['partido_nombre'].nunique()}")

# Validación: ¿Se quedaron partidos sin nombre?
nulos = se_candidatos_municipio['partido_nombre'].isna().sum()
if nulos > 0:
    print(f"\n⚠️ OJO: Hay {nulos} registros donde el código de partido no tiene nombre asignado.")

print("\nFilas por circunscripción:")
print(se_candidatos_municipio.groupby('circunscripcion').size())

print(f"\nColumnas finales: {list(se_candidatos_municipio.columns)}")
se_candidatos_municipio.head(10)

se_candidatos_municipio: 1,304,333 filas × 18 columnas
  Departamentos: 34
  Municipios: 1100
  Circunscripciones: 2
  Candidatos únicos: 1072
  Partidos: 26

Filas por circunscripción:
circunscripcion
Indígenas      39237
Nacional     1265096
dtype: int64

Columnas finales: ['depto_nombre', 'mpio_codigo', 'mpio_nombre', 'circunscripcion_id', 'circunscripcion', 'partido_codigo', 'votos_partido_mpio', 'candidato_codigo', 'candidato_cedula', 'candidato_nombre', 'candidato_votos', 'candidato_porcentaje', 'preferido', 'mpio_votantes', 'mpio_votos_validos', 'mpio_votos_nulos', 'mpio_votos_no_marcados', 'partido_nombre']


,depto_nombre,mpio_codigo,mpio_nombre,circunscripcion_id,circunscripcion,partido_codigo,votos_partido_mpio,candidato_codigo,candidato_cedula,candidato_nombre,candidato_votos,candidato_porcentaje,preferido,mpio_votantes,mpio_votos_validos,mpio_votos_nulos,mpio_votos_no_marcados,partido_nombre
0,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,0,,SOLO POR LA LISTA,5143,"19,11%",0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
1,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,1,43913613,DIANA CAROLINA CORCHO MEJIA,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
2,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,2,72212951,PEDRO HERNANDO FLOREZ PORRAS,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
3,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,3,40926458,CARMEN PATRICIA CAICEDO OMAR,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
4,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,4,16823075,WILSON NEBER ARIAS CASTILLO,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
5,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,5,1032411209,LAURA CRISTINA AHUMADA GARCIA,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
6,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,6,1057585551,WALTER ALFONSO RODRIGUEZ CHAPARRO,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
7,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,7,41391027,AIDA YOLANDA AVELLA ESQUIVEL,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
8,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,8,10487216,FERNEY SILVA IDROBO,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO
9,CESAR,1200150,AGUSTIN CODAZZI,0,Nacional,92,5143,9,1032365459,YULY ESMERALDA HERNANDEZ SILVA,0,0%,0,29414,26904,1495,1015,PACTO HISTÓRICO SENADO


In [23]:
# === Resumen: Top candidatos por circunscripción (agregado desde municipios) ===
for circ in ['Nacional', 'Indígenas']:
    subset = se_candidatos_municipio[se_candidatos_municipio['circunscripcion'] == circ]
    resumen = subset.groupby(['partido_nombre', 'candidato_nombre']).agg(
        votos_totales=('candidato_votos', 'sum'),
        mpios_presencia=('mpio_codigo', 'nunique'),
    ).sort_values('votos_totales', ascending=False).reset_index()
    
    print(f"\n{'='*90}")
    print(f"Top 10 - {circ} ({len(resumen)} candidatos, {subset['candidato_votos'].sum():,} votos)")
    print(f"{'='*90}")
    for _, row in resumen.head(10).iterrows():
        print(f"  {row['votos_totales']:>10,}  {row['candidato_nombre'][:40]:<42s} ({row['partido_nombre'][:35]}) [{row['mpios_presencia']} mpios]")


Top 10 - Nacional (1064 candidatos, 18,806,189 votos)
   4,413,636  SOLO POR LA LISTA                          (PACTO HISTÓRICO SENADO) [1189 mpios]
   3,035,715  SOLO POR LA LISTA                          (PARTIDO CENTRO DEMOCRÁTICO) [1189 mpios]
     216,376  SOLO POR LA LISTA                          (PARTIDO LIBERAL COLOMBIANO) [1189 mpios]
     182,596  SOLO POR LA LISTA                          (PARTIDO CONSERVADOR COLOMBIANO) [1189 mpios]
     178,907  NADYA GEORGETTE BLEL SCAFF                 (PARTIDO CONSERVADOR COLOMBIANO) [1189 mpios]
     175,388  LIDIO ARTURO GARCIA TURBAY                 (PARTIDO LIBERAL COLOMBIANO) [1189 mpios]
     166,070  SOLO POR LA LISTA                          (ALIANZA POR COLOMBIA) [1189 mpios]
     159,956  JONATHAN FERNEY PULIDO HERNANDEZ           (ALIANZA POR COLOMBIA) [1189 mpios]
     146,563  SOLO POR LA LISTA                          (PARTIDO DE LA UNIÓN POR LA GENTE - ) [1189 mpios]
     146,507  YESSID ENRIQUE PULGAR DAZA             

## Exportación de datos (máxima granularidad municipal)

Exportamos `se_candidatos_municipio` — votos de cada candidato de cada partido en cada municipio, por cada circunscripción — en 5 formatos a `data/raw_data_extract/senado/`.

| Formato | Archivo | Uso típico |
|---------|---------|------------|
| CSV (coma) | `.csv` | Compatible universal |
| CSV (punto y coma) | `_semicolon.csv` | Excel con coma decimal |
| Excel | `.xlsx` | Hojas de cálculo |
| JSON | `.json` | APIs, visualizaciones web |
| Parquet | `.parquet` | Alto rendimiento, Spark, pandas |

In [24]:
import os

OUTPUT_DIR = os.path.join('..', '..', 'data', 'raw_data_extract', 'senado')
os.makedirs(OUTPUT_DIR, exist_ok=True)

base = os.path.join(OUTPUT_DIR, 'se_candidatos_municipio')

# CSV (separado por coma)
se_candidatos_municipio.to_csv(f'{base}.csv', index=False, encoding='utf-8-sig')

# CSV (separado por punto y coma)
se_candidatos_municipio.to_csv(f'{base}_semicolon.csv', index=False, encoding='utf-8-sig', sep=';')

# Excel (skip if > 1,048,576 rows — Excel limit)
if len(se_candidatos_municipio) <= 1_048_576:
    se_candidatos_municipio.to_excel(f'{base}.xlsx', index=False, engine='openpyxl')
else:
    print(f"⚠️  XLSX omitido: {len(se_candidatos_municipio):,} filas excede el límite de Excel (1,048,576)")

# JSON
se_candidatos_municipio.to_json(f'{base}.json', orient='records', force_ascii=False, indent=2)

# Parquet
se_candidatos_municipio.to_parquet(f'{base}.parquet', index=False, engine='pyarrow')

formatos = ['.csv', '_semicolon.csv', '.json', '.parquet']
if len(se_candidatos_municipio) <= 1_048_576:
    formatos.insert(2, '.xlsx')
print(f"\n✅ se_candidatos_municipio ({len(se_candidatos_municipio):,} filas) — {len(formatos)} archivos exportados:")
for ext in formatos:
    filepath = f'{base}{ext}'
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"   {os.path.basename(filepath):50s} {size_mb:.2f} MB")

⚠️  XLSX omitido: 1,304,333 filas excede el límite de Excel (1,048,576)

✅ se_candidatos_municipio (1,304,333 filas) — 4 archivos exportados:
   se_candidatos_municipio.csv                        171.44 MB
   se_candidatos_municipio_semicolon.csv              170.88 MB
   se_candidatos_municipio.json                       730.61 MB
   se_candidatos_municipio.parquet                    5.11 MB
